# 04 - Model Training and Performance

Purpose: train candidate models, register the champion in MLflow, and review test/OOT performance.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyspark
from pyspark.sql import SparkSession, functions as F, Window

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATAMART_DIR = PROJECT_ROOT / "datamart"
MODEL_BANK_DIR = PROJECT_ROOT / "outputs" / "model_bank"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

def spark_path(path):
    return Path(path).resolve().as_posix()

spark = (pyspark.sql.SparkSession.builder
         .appName("assignment_2_notebook")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print(f"Project root: {PROJECT_ROOT}")
print(f"Spark version: {spark.version}")

In [ ]:
RUN_TRAINING = True

feature_store = spark.read.parquet(spark_path(DATAMART_DIR / "gold" / "feature_store"))
print(f"Feature store rows: {feature_store.count():,}")
display(feature_store.groupBy("feature_snapshot_date")
        .agg(F.count("*").alias("row_count"), F.avg("label").alias("default_rate"))
        .orderBy("feature_snapshot_date").toPandas())

In [ ]:
if RUN_TRAINING:
    from utils.model_training import train_and_register_model
    metadata = train_and_register_model()
else:
    with open(MODEL_BANK_DIR / "champion_metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

print(json.dumps({"model_name": metadata["model_name"],
                  "model_version": metadata["model_version"],
                  "split_strategy": metadata["split_strategy"],
                  "decision_threshold": metadata["decision_threshold"]}, indent=2))

In [ ]:
split_windows = pd.DataFrame(metadata["split_windows"]).T.reset_index().rename(columns={"index": "dataset_split"})
display(split_windows)

class_balance = pd.DataFrame([metadata["class_balance"]]).T.reset_index()
class_balance.columns = ["metric", "value"]
display(class_balance)

In [ ]:
train_metrics = (pd.DataFrame(metadata.get("candidate_train_metrics", {})).T.reset_index().rename(columns={"index": "model_name"}).assign(dataset_split="train"))
test_metrics = (pd.DataFrame(metadata["candidate_test_metrics"]).T.reset_index().rename(columns={"index": "model_name"}).assign(dataset_split="test"))
oot_metrics = (pd.DataFrame(metadata["candidate_oot_metrics"]).T.reset_index().rename(columns={"index": "model_name"}).assign(dataset_split="oot"))
metrics = pd.concat([train_metrics, test_metrics, oot_metrics], ignore_index=True)
display(metrics.sort_values(["dataset_split", "roc_auc"], ascending=[True, False]))

In [ ]:
plot_metrics = metrics.melt(id_vars=["model_name", "dataset_split"], value_vars=["roc_auc", "average_precision", "precision", "recall", "f1"], var_name="metric", value_name="value",)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=plot_metrics, x="metric", y="value", hue="model_name", ax=ax)
ax.set_title("Candidate model performance")
ax.set_ylim(0, 1)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

In [ ]:
history_path = MODEL_BANK_DIR / "metrics_history.csv"
if history_path.exists():
    history = spark.read.option("header", True).option("inferSchema", True).csv(spark_path(history_path))
    display(history.orderBy(F.desc("model_version"), "dataset_split", F.desc("roc_auc")).toPandas())
else:
    print("No metrics history file found yet.")

Champion selection rule: the pipeline selects the model with the best test ROC AUC, then reports OOT metrics separately to show forward-time generalization.